In [1]:
import sys, random, math
from collections import Counter
import numpy as np

from framework.Tensor import Tensor
from framework.CrossEntropyLoss import CrossEntropyLoss
from framework.SGD import SGD
from framework.Embedding import Embedding
from framework.RNNCell import RNNCell

In [2]:
f = open('tasks_1-20_v1-2/en/qa1_single-supporting-fact_train.txt', 'r')
raw = f.readlines()
f.close()

In [3]:
tokens = list()
for line in raw[0:1000]:
    tokens.append(line.lower().replace("\n", "").split(" ")[1:])

In [4]:
new_tokens = list()
for line in tokens:
    new_tokens.append(['-'] * (6 - len(line)) + line)
tokens = new_tokens

In [5]:
vocab = set()
for sent in tokens:
    for word in sent:
        vocab.add(word)

In [6]:
vocab = list(vocab)

In [7]:
word2index = {}
for i, word in enumerate(vocab):
    word2index[word] = i

In [8]:
def words2indices(sentence):
    idx = list()
    for word in sentence:
        idx.append(word2index[word])
    return idx

In [9]:
indices = list()
for line in tokens:
    idx = list()
    for w in line:
        idx.append(word2index[w])
    indices.append(idx)

In [10]:
data = np.array(indices)

In [16]:
embed = Embedding(vocab_size=len(vocab), dim=16)
model = RNNCell(n_inputs=16, n_hidden=16, n_output=len(vocab))

loss = CrossEntropyLoss()
params = model.get_params() + embed.get_params()
optimizer = SGD(params=params, alpha=0.1)

for iter in range(1000):
    batch_size = 50
    total_loss = 0

    hidden = model.init_hidden(batch_size=batch_size)

    for t in range(5):
        input = Tensor(data[0:batch_size, t], autograd=True)
        rnn_input = embed.forward(input=input)
        output, hidden = model.forward(input=rnn_input, hidden=hidden)

    target = Tensor(data[0:batch_size, t + 1], autograd=True)
    l = loss.forward(output, target)
    l.backward()
    optimizer.step()
    total_loss += l.data
    if (iter % 200 == 0):
        p_correct = (target.data == np.argmax(output.data, axis=1)).mean()
        print_loss = total_loss / (len(data) / batch_size)
        print("Loss:", print_loss, "% Correct: ", p_correct)

Loss: 0.22484383023515706 % Correct:  0.12
Loss: 0.07878373655809387 % Correct:  0.3
Loss: 0.07704158639320273 % Correct:  0.34
Loss: 0.07106254040891194 % Correct:  0.4
Loss: 0.06603238455725888 % Correct:  0.4


In [18]:
batch_size = 1
hidden = model.init_hidden(batch_size=batch_size)
for t in range(5):
    input = Tensor(data[0:batch_size, t], autograd=True)
    rnn_input = embed.forward(input=input)
    output, hidden = model.forward(input=rnn_input, hidden=hidden)

In [19]:
target = Tensor(data[0:batch_size, t + 1], autograd=True)
l = loss.forward(output, target)

In [20]:
ctx = ""
for idx in data[0:batch_size][0][0:-1]:
    ctx += vocab[idx] + " "

In [21]:
print("Context: ", ctx)
print("Pred: ", vocab[output.data.argmax()])

Context:  - mary moved to the 
Pred:  office.
